# AutoML for Time Series — AutoGluon TimeSeries
## Data Loading

## Packages

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

In [ ]:
data_dir = os.path.dirname(os.path.abspath("AutoGluonTS.ipynb"))

# --- Raw files ---
raw_files = sorted(glob.glob(os.path.join(data_dir, "EF* - OD.csv")))

dfs_raw = {}
for path in raw_files:
    station = os.path.basename(path).split(" - ")[0]  # e.g. "EF01"
    df = pd.read_csv(
        path,
        sep=";",
        decimal=",",
        parse_dates=["Data"],
        dayfirst=True,
    )
    df = df.rename(columns={"Data": "timestamp"})
    df["station"] = station
    dfs_raw[station] = df
    print(f"{station} raw: {df.shape}  |  {df['timestamp'].min()} → {df['timestamp'].max()}")

print()

# --- Complete series files ---
complete_series_files = sorted(glob.glob(os.path.join(data_dir, "*_DO_complete_series.csv")))

dfs_complete_series = {}
for path in complete_series_files:
    station = os.path.basename(path).split("_")[0]  # e.g. "EF01"
    df = pd.read_csv(
        path,
        sep=";",
        decimal=",",
        parse_dates=["Data"],
    )
    df = df.rename(columns={"Data": "timestamp", "DO_filled_mgL": "DO_mgL"})
    df["station"] = station
    dfs_complete_series[station] = df
    print(f"{station} complete_series: {df.shape}  |  {df['timestamp'].min()} → {df['timestamp'].max()}")

# --- Fallback: imputed files (for stations without complete_series, e.g. EF03) ---
imputed_files = sorted(glob.glob(os.path.join(data_dir, "*_DO_imputed.csv")))
for path in imputed_files:
    station = os.path.basename(path).split("_")[0]
    if station in dfs_complete_series:
        continue  # already has complete_series
    df = pd.read_csv(
        path,
        sep=";",
        decimal=",",
        parse_dates=["Data"],
    )
    df = df.rename(columns={"Data": "timestamp", "DO_imputed_mgL": "DO_mgL"})
    df["station"] = station
    dfs_complete_series[station] = df
    print(f"{station} imputed (fallback): {df.shape}  |  {df['timestamp'].min()} → {df['timestamp'].max()}")

## Data Preparation for AutoGluon TimeSeries

### Constants and Temporal Split

In [ ]:
START_DATE        = "2015-01-01"
CUTOFF            = pd.Timestamp("2025-01-01")   # start of test period
CUTOFF_END        = pd.Timestamp("2025-04-01")   # end of test period (exclusive)
PREDICTION_LENGTH = int((CUTOFF_END - CUTOFF).total_seconds() // 3600)

print(f"Test period      : {CUTOFF.date()} → {(CUTOFF_END - pd.Timedelta(hours=1)).date()}")
print(f"Prediction length: {PREDICTION_LENGTH} h  ({PREDICTION_LENGTH // 24} days)")

### Hourly Series: Raw and Complete Series Versions

In [ ]:
series_raw     = {}
series_complete_series = {}

for station in sorted(dfs_raw):
    if station not in dfs_complete_series:
        print(f"{station}: no imputed data available, skipping.")
        continue

    # ── RAW ──────────────────────────────────────────────────────────────
    raw = dfs_raw[station][["timestamp", "Media"]].copy()
    raw["DO_mgL"] = pd.to_numeric(raw["Media"], errors="coerce")
    raw = raw[["timestamp", "DO_mgL"]].set_index("timestamp")

    # ── IMPUTED ──────────────────────────────────────────────────────────
    imp = dfs_complete_series[station].set_index("timestamp")[["DO_mgL"]].copy()

    # ── Common window (ensures same start and end) ─────────────────────
    common_start = max(raw.index.min(), imp.index.min(), pd.Timestamp(START_DATE))
    common_end   = min(raw.index.max(), imp.index.max())

    raw = raw[(raw.index >= common_start) & (raw.index <= common_end)]
    imp = imp[(imp.index >= common_start) & (imp.index <= common_end)]

    # ── Full hourly grid ────────────────────────────────────────────────
    full_idx = pd.date_range(common_start, common_end, freq="h")

    raw = raw.reindex(full_idx); raw.index.name = "timestamp"
    imp = imp.reindex(full_idx); imp.index.name = "timestamp"

    series_raw[station]     = raw
    series_complete_series[station] = imp

    print(f"{station}  window: {common_start.date()} → {common_end.date()}")
    print(f"  raw             : {len(raw):,} pts | NaN: {raw['DO_mgL'].isna().sum():,}")
    print(f"  complete_series : {len(imp):,} pts | NaN: {imp['DO_mgL'].isna().sum():,}")
    print()

In [ ]:
# ── Baseline I  — ffill().bfill() ────────────────────────────────────────────
# Propagates the last observed value forward (and the first backward for leading NaN).
# Fast and parameter-free, but does not respect local trends.
# Filling restricted to training period (< CUTOFF) to avoid data leakage.
series_baseline_I = {}
for station, raw in series_raw.items():
    base = raw.copy()
    mask = base.index < CUTOFF
    base.loc[mask, "DO_mgL"] = base.loc[mask, "DO_mgL"].ffill().bfill()
    series_baseline_I[station] = base
    print(f"{station}  Baseline I  — training NaN: {base.loc[mask, 'DO_mgL'].isna().sum():,}")

print()

# ── Baseline II — linear interpolation ───────────────────────────────────────
# Fills internal gaps with linear interpolation between the two nearest observed
# neighbours; ffill/bfill only at boundaries without prior/posterior neighbours.
# Respects local trend but assumes linearity between known points.
# Filling restricted to training period (< CUTOFF) to avoid data leakage.
series_baseline_II = {}
for station, raw in series_raw.items():
    base = raw.copy()
    mask = base.index < CUTOFF
    filled = (
        base.loc[mask, "DO_mgL"]
        .interpolate(method="linear")   # interior: linear between neighbours
        .ffill()                         # leading NaN without prior neighbour
        .bfill()                         # residual trailing NaN
    )
    base.loc[mask, "DO_mgL"] = filled
    series_baseline_II[station] = base
    print(f"{station}  Baseline II — training NaN: {base.loc[mask, 'DO_mgL'].isna().sum():,}")

### Differences Between Baselines

| | **Baseline I** (ffill/bfill) | **Baseline II** (linear interpolation) | **Hierarchical Filling** |
|---|---|---|---|
| **Method** | Copies the last observed value forward; the first backward for leading NaN | Linearly interpolates between the two nearest observed neighbours of each gap | Hierarchical pipeline: Interp. (≤10 h) → MA (≤42 h) → STL (≤72 h) → Prophet (> 72 h) |
| **Local trend** | No — holds the value constant throughout the gap | Yes — linear transition between neighbouring points | Yes — captures daily/weekly/annual seasonality and long-term trend |
| **Long gaps** | May propagate a value for days or weeks | May produce unrealistic ramps over very long gaps | Uses STL/Prophet which model seasonal patterns |
| **QC / constancy** | No — uses raw series with original NaN | No — uses raw series with original NaN | Yes — removes physical outliers and flatline periods before imputation |
| **Role here** | Lower reference bound — trivial strategy | Upper reference bound — classical strategy | Treated series under evaluation |
| **Anti-leakage** | Filling restricted to `index < CUTOFF`; test evaluated against original raw series | Same | External file generated before the test period |

In [ ]:
# ── Save baseline series to CSV for use in other notebooks ────────────────────
# Filling is applied to the FULL series (train + test) here, as these CSVs
# are intended for other contexts that will enforce their own leakage control.
# Format compatible with *_DO_complete_series.csv files from the pipeline.

for station, raw in series_raw.items():
    full = raw["DO_mgL"].copy()

    # Baseline I: ffill/bfill global
    bI = full.ffill().bfill()
    dfI = pd.DataFrame({"Data": bI.index, "DO_filled_mgL": bI.values})
    pathI = os.path.join(data_dir, f"{station}_DO_baseline_I.csv")
    dfI.to_csv(pathI, sep=";", decimal=",", index=False)

    # Baseline II: global linear interpolation
    bII = full.interpolate(method="linear").ffill().bfill()
    dfII = pd.DataFrame({"Data": bII.index, "DO_filled_mgL": bII.values})
    pathII = os.path.join(data_dir, f"{station}_DO_baseline_II.csv")
    dfII.to_csv(pathII, sep=";", decimal=",", index=False)

    nan_I  = bI.isna().sum()
    nan_II = bII.isna().sum()
    print(f"{station}  → {os.path.basename(pathI)}  (NaN: {nan_I})  |  "
          f"{os.path.basename(pathII)}  (NaN: {nan_II})")

print("\nCSVs saved. Columns: Data, DO_filled_mgL")

### TimeSeriesDataFrame and Train/Test Split

In [ ]:
def build_ts(series_dict):
    """Convert dict {station: DataFrame} to TimeSeriesDataFrame."""
    frames = []
    for station, df in series_dict.items():
        tmp = df.reset_index()
        tmp["item_id"] = station
        frames.append(tmp)
    combined = pd.concat(frames, ignore_index=True)
    return TimeSeriesDataFrame.from_data_frame(
        combined, id_column="item_id", timestamp_column="timestamp"
    )

def split(ts):
    """Temporal split: train < CUTOFF, test in [CUTOFF, CUTOFF_END)."""
    train = ts[ts.index.get_level_values("timestamp") < CUTOFF]
    test  = ts[
        (ts.index.get_level_values("timestamp") >= CUTOFF) &
        (ts.index.get_level_values("timestamp") <  CUTOFF_END)
    ]
    return train, test

# ts_raw kept as ground truth only — not included in DATASETS
ts_raw              = build_ts(series_raw)
ts_complete_series  = build_ts(series_complete_series)
ts_baseline_I       = build_ts(series_baseline_I)
ts_baseline_II      = build_ts(series_baseline_II)

train_raw,             test_raw             = split(ts_raw)
train_complete_series, test_complete_series = split(ts_complete_series)
train_baseline_I,      test_baseline_I      = split(ts_baseline_I)
train_baseline_II,     test_baseline_II     = split(ts_baseline_II)

DATASETS = {
    "baseline_I" : {"train": train_baseline_I,  "test": test_baseline_I,  "full": ts_baseline_I},
    "baseline_II": {"train": train_baseline_II, "test": test_baseline_II, "full": ts_baseline_II},
    "complete_series": {"train": train_complete_series, "test": test_complete_series, "full": ts_complete_series},
}

for name, d in DATASETS.items():
    print(f"\n── {name} ──")
    print(f"  train : {d['train'].index.get_level_values('timestamp').min()} → "
          f"{d['train'].index.get_level_values('timestamp').max()}")
    print(f"  test  : {d['test'].index.get_level_values('timestamp').min()} → "
          f"{d['test'].index.get_level_values('timestamp').max()}")
    print(f"  train rows: {len(d['train']):,}  | test rows: {len(d['test']):,}")

## Modelling — DeepAR · XGBoost · Prophet · SARIMA

In [6]:
MODELS_CFG = {
    "DeepAR": {
        "DeepAR": {
            "hidden_size": 64,
            "num_layers" : 2,
            "max_epochs" : 50,
            "context_length": 24 * 7,
        }
    },
    "XGBoost": {
        "DirectTabular": {
            "tabular_hyperparameters": {
                "XGB": {"n_estimators": 300, "learning_rate": 0.05, "random_state": 42}
            }
        }
    },
}


In [ ]:
# ── Seasonally scaled metrics (m = 24 h — daily seasonality) ─────────────────
METRIC_M = 24

def _scale_mae(y_train, m=METRIC_M):
    y = np.asarray(y_train, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) <= m:
        return np.nan
    return np.mean(np.abs(y[m:] - y[:-m]))

def _scale_rmse(y_train, m=METRIC_M):
    y = np.asarray(y_train, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) <= m:
        return np.nan
    return np.sqrt(np.mean((y[m:] - y[:-m]) ** 2))

def mase(obs, pred, y_train, m=METRIC_M):
    s = _scale_mae(y_train, m)
    if not s or np.isnan(s):
        return np.nan
    return np.mean(np.abs(obs - pred)) / s

def rmsse(obs, pred, y_train, m=METRIC_M):
    s = _scale_rmse(y_train, m)
    if not s or np.isnan(s):
        return np.nan
    return np.sqrt(np.mean((obs - pred) ** 2)) / s

def r2_score(obs, pred):
    ss_tot = np.sum((obs - obs.mean()) ** 2)
    if ss_tot == 0:
        return np.nan
    return 1 - np.sum((obs - pred) ** 2) / ss_tot

def da(obs, pred):
    """Directional Accuracy — fraction with correct direction (non-flat obs only)."""
    d_obs, d_pred = np.diff(obs), np.diff(pred)
    mask = d_obs != 0
    if not mask.any():
        return np.nan
    return float(np.mean(np.sign(d_obs[mask]) == np.sign(d_pred[mask])))

def mda(obs, pred):
    """Mean Directional Accuracy — all steps including flat."""
    d_obs, d_pred = np.diff(obs), np.diff(pred)
    if len(d_obs) == 0:
        return np.nan
    return float(np.mean(np.sign(d_obs) == np.sign(d_pred)))

def maed(obs, pred):
    """Mean Absolute Error on First Differences."""
    if len(obs) < 2:
        return np.nan
    return float(np.mean(np.abs(np.diff(obs) - np.diff(pred))))

def ce_trend(obs, pred, n_bins=3, eps=1e-9):
    """Cross-Entropy on Quantized Trends (down / flat / up)."""
    d_obs, d_pred = np.diff(obs), np.diff(pred)
    if len(d_obs) == 0:
        return np.nan
    nz = d_obs[d_obs != 0]
    lim = float(np.percentile(np.abs(nz), 90)) if len(nz) > 0 else 1e-6
    edges = np.array([-np.inf, -lim, lim, np.inf])
    p = np.histogram(d_obs,  bins=edges)[0] / len(d_obs)
    q = np.histogram(d_pred, bins=edges)[0] / len(d_pred)
    q = np.clip(q, eps, 1); q /= q.sum()
    return float(-np.sum(p * np.log(q)))

In [ ]:
import time
fit_times = {}  # {ds_name: {model_name: seconds}}

# predictors[dataset_name][model_name] = trained TimeSeriesPredictor
predictors = {}

for ds_name, ds in DATASETS.items():
    predictors[ds_name] = {}
    for model_name, hyp in MODELS_CFG.items():
        print(f"\n{'─'*60}")
        print(f"  Dataset: {ds_name}  |  Model: {model_name}")
        print(f"{'─'*60}")
        pred = TimeSeriesPredictor(
            target="DO_mgL",
            prediction_length=PREDICTION_LENGTH,
            freq="h",
            eval_metric="MASE",
            path=f"AutoGluon_{ds_name}_{model_name}",
        )
        t0 = time.time()
        pred.fit(ds["train"], hyperparameters=hyp, time_limit=1800, verbosity=1)
        fit_times.setdefault(ds_name, {})[model_name] = time.time() - t0
        predictors[ds_name][model_name] = pred
        print(f"  ✓ {ds_name} / {model_name} done.")

### Prophet (Facebook Prophet — manual implementation)

In [ ]:
import time

from prophet import Prophet

prophet_forecasts  = {}   # prophet_forecasts[ds_name][station] = pd.Series (yhat, index=timestamp)
prophet_metric_rows = []

for ds_name, ds in DATASETS.items():
    prophet_forecasts[ds_name] = {}
    for sid in sorted(ds["train"].item_ids):
        print(f"  Training Prophet | {ds_name} | {sid} ...", end=" ", flush=True)

        # Training data for this station
        train_s = (
            pd.DataFrame(ds["train"].loc[sid][["DO_mgL"]].reset_index())
            .rename(columns={"timestamp": "ds", "DO_mgL": "y"})
            .dropna()
        )

        m = Prophet(
            seasonality_mode="additive",
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=True,
        )
        t0_fit = time.time()
        m.fit(train_s)
        fit_t = time.time() - t0_fit

        # Forecast over the test period
        future = pd.DataFrame({
            "ds": pd.date_range(CUTOFF, CUTOFF_END - pd.Timedelta(hours=1), freq="h")
        })
        t0_pred = time.time()
        fc = m.predict(future).set_index("ds")["yhat"]
        pred_t = time.time() - t0_pred
        fc.index.name = "timestamp"
        prophet_forecasts[ds_name][sid] = fc

        # Metrics against test_raw
        if sid in test_raw.item_ids:
            obs    = test_raw.loc[sid]["DO_mgL"]
            pred_v_s = fc.reindex(obs.index)
            mask   = obs.notna() & pred_v_s.notna()
            obs_v  = obs[mask].values
            pred_v = pred_v_s[mask].values
            if len(obs_v):
                y_tr_v = ds["train"].loc[sid]["DO_mgL"].values

                mase_v  = mase(obs_v, pred_v, y_tr_v)
                rmsse_v = rmsse(obs_v, pred_v, y_tr_v)
                r2_v    = r2_score(obs_v, pred_v)
                da_v      = da(obs_v, pred_v)
                mda_v     = mda(obs_v, pred_v)
                maed_v    = maed(obs_v, pred_v)
                cetrend_v = ce_trend(obs_v, pred_v)

                prophet_metric_rows.append({
                    "Model"   : "Prophet",
                    "Dataset" : ds_name,
                    "Station" : sid,
                    "MASE"    : mase_v,
                    "RMSSE"   : rmsse_v,
                    "R²"      : r2_v,
                    "DA"      : da_v,
                    "MDA"     : mda_v,
                    "MAED"    : maed_v,
                    "CETrend" : cetrend_v,
                    "Fit_s"   : round(fit_t, 2),
                    "Pred_s"  : round(pred_t, 2),
                })
        print("✓")

prophet_df = pd.DataFrame(prophet_metric_rows)

print(f"\n{'═'*70}")
print("  Model: Prophet")
print(f"{'═'*70}")

table = prophet_df.pivot_table(
    index="Station",
    columns="Dataset",
    values=["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"],
    aggfunc="first",
)
table = table.reindex(columns=pd.MultiIndex.from_product(
    [["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"], sorted(DATASETS.keys())]
))
table.columns = [f"{m} [{d}]" for m, d in table.columns]
table.loc["MEAN"] = table.mean()
display(table.round(4))

csv_path = os.path.join(data_dir, "AutoGluonTS_metrics_Prophet.csv")
table.round(4).to_csv(csv_path, sep=";", decimal=",")
print(f"  → Saved: {csv_path}")

### SARIMA (statsmodels — manual implementation)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Recent training window: SARIMA is infeasible on 87k hourly points
SARIMA_WINDOW = 24 * 60          # last 60 days
SARIMA_ORDER          = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (0, 1, 1, 24)

sarima_forecasts   = {}
sarima_metric_rows = []

for ds_name, ds in DATASETS.items():
    sarima_forecasts[ds_name] = {}
    for sid in sorted(ds["train"].item_ids):
        print(f"  Training SARIMA | {ds_name} | {sid} ...", end=" ", flush=True)

        # Training series (recent window, no NaN)
        y_full  = ds["train"].loc[sid]["DO_mgL"].ffill().bfill()
        y_train = y_full.iloc[-SARIMA_WINDOW:]

        try:
            t0_fit = time.time()
            result = SARIMAX(
                y_train,
                order=SARIMA_ORDER,
                seasonal_order=SARIMA_SEASONAL_ORDER,
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False)
            fit_t = time.time() - t0_fit

            t0_pred = time.time()
            fc = result.forecast(steps=PREDICTION_LENGTH)
            pred_t = time.time() - t0_pred
            fc.index = pd.date_range(CUTOFF, periods=PREDICTION_LENGTH, freq="h")
            fc.index.name = "timestamp"
            sarima_forecasts[ds_name][sid] = fc

            # Metrics against test_raw
            if sid in test_raw.item_ids:
                obs      = test_raw.loc[sid]["DO_mgL"]
                pred_s   = fc.reindex(obs.index)
                mask     = obs.notna() & pred_s.notna()
                obs_v    = obs[mask].values
                pred_v   = pred_s[mask].values
                if len(obs_v):
                    y_tr_v = ds["train"].loc[sid]["DO_mgL"].values

                    mase_v  = mase(obs_v, pred_v, y_tr_v)
                    rmsse_v = rmsse(obs_v, pred_v, y_tr_v)
                    r2_v    = r2_score(obs_v, pred_v)
                    da_v      = da(obs_v, pred_v)
                    mda_v     = mda(obs_v, pred_v)
                    maed_v    = maed(obs_v, pred_v)
                    cetrend_v = ce_trend(obs_v, pred_v)

                    sarima_metric_rows.append({
                        "Model"   : "SARIMA",
                        "Dataset" : ds_name,
                        "Station" : sid,
                        "MASE"    : mase_v,
                        "RMSSE"   : rmsse_v,
                        "R²"      : r2_v,
                        "DA"      : da_v,
                        "MDA"     : mda_v,
                        "MAED"    : maed_v,
                        "CETrend" : cetrend_v,
                        "Fit_s"   : round(fit_t, 2),
                        "Pred_s"  : round(pred_t, 2),
                    })
            print("✓")
        except Exception as e:
            print(f"✗ ERROR: {e}")
            sarima_forecasts[ds_name][sid] = None

sarima_df = pd.DataFrame(sarima_metric_rows)

print(f"\n{'═'*70}")
print("  Model: SARIMA")
print(f"{'═'*70}")

table = sarima_df.pivot_table(
    index="Station",
    columns="Dataset",
    values=["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"],
    aggfunc="first",
)
table = table.reindex(columns=pd.MultiIndex.from_product(
    [["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"], sorted(DATASETS.keys())]
))
table.columns = [f"{m} [{d}]" for m, d in table.columns]
table.loc["MEAN"] = table.mean()
display(table.round(4))

csv_path = os.path.join(data_dir, "AutoGluonTS_metrics_SARIMA.csv")
table.round(4).to_csv(csv_path, sep=";", decimal=",")
print(f"  → Saved: {csv_path}")

## Evaluation and Comparison

In [ ]:
pred_times = {}  # {ds_name: {model_name: seconds}}
rows = []
forecasts = {}   # forecasts[ds_name][model_name] = predictions

for ds_name, ds in DATASETS.items():
    forecasts[ds_name] = {}
    for model_name, pred in predictors[ds_name].items():
        # ground truth always = original raw, for fair comparison between datasets
        lb    = pred.leaderboard(ts_raw, silent=True)
        score = lb["score_test"].iloc[0]
        rows.append({
            "Dataset": ds_name,
            "Model"  : model_name,
            "MASE"   : round(score, 4),
        })
        t0 = time.time()
        forecasts[ds_name][model_name] = pred.predict(ds["train"])
        pred_times.setdefault(ds_name, {})[model_name] = time.time() - t0

comparison = (
    pd.DataFrame(rows)
    .pivot(index="Model", columns="Dataset", values="MASE")
    .sort_values("complete_series")
)
print(comparison.to_string())

In [ ]:
metric_rows = []

for ds_name in DATASETS:
    for model_name, fc in forecasts[ds_name].items():
        for sid in ts_raw.item_ids:
            if sid not in test_raw.item_ids:
                continue
            obs = test_raw.loc[sid]["DO_mgL"]

            if sid not in fc.item_ids:
                continue
            pred = fc.loc[sid]["mean"].reindex(obs.index)

            mask = obs.notna() & pred.notna()
            obs_v  = obs[mask].values
            pred_v = pred[mask].values

            if len(obs_v) == 0:
                continue

            y_tr_v = DATASETS[ds_name]["train"].loc[sid]["DO_mgL"].values

            mase_v  = mase(obs_v, pred_v, y_tr_v)
            rmsse_v = rmsse(obs_v, pred_v, y_tr_v)
            r2_v    = r2_score(obs_v, pred_v)
            da_v      = da(obs_v, pred_v)
            mda_v     = mda(obs_v, pred_v)
            maed_v    = maed(obs_v, pred_v)
            cetrend_v = ce_trend(obs_v, pred_v)

            n_s = len(ts_raw.item_ids)
            metric_rows.append({
                "Model"   : model_name,
                "Dataset" : ds_name,
                "Station" : sid,
                "MASE"    : mase_v,
                "RMSSE"   : rmsse_v,
                "R²"      : r2_v,
                "DA"      : da_v,
                "MDA"     : mda_v,
                "MAED"    : maed_v,
                "CETrend" : cetrend_v,
                "Fit_s"   : round(fit_times.get(ds_name, {}).get(model_name, np.nan) / n_s, 2),
                "Pred_s"  : round(pred_times.get(ds_name, {}).get(model_name, np.nan) / n_s, 2),
            })

metrics_df = pd.DataFrame(metric_rows)

# ── Per-model table ───────────────────────────────────────────────────────────
for model_name in MODELS_CFG:
    print(f"\n{'═'*70}")
    print(f"  Model: {model_name}")
    print(f"{'═'*70}")

    sub = metrics_df[metrics_df["Model"] == model_name].copy()

    table = sub.pivot_table(
        index="Station",
        columns="Dataset",
        values=["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"],
        aggfunc="first",
    )

    table = table.reindex(columns=pd.MultiIndex.from_product(
        [["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"], sorted(DATASETS.keys())]
    ))
    table.columns = [f"{m} [{d}]" for m, d in table.columns]

    table.loc["MEAN"] = table.mean()

    display(table.round(4))

    csv_path = os.path.join(data_dir, f"AutoGluonTS_metrics_{model_name}.csv")
    table.round(4).to_csv(csv_path, sep=";", decimal=",")
    print(f"  → Saved: {csv_path}")

In [ ]:
import matplotlib.pyplot as plt
import os

DS_ORDER = ["baseline_I", "baseline_II", "complete_series"]
DS_LABELS = {
    "baseline_I":       "Baseline I",
    "baseline_II":      "Baseline II",
    "complete_series":  "Hierarchical Filling",
}

CONTEXT_DAYS = 30  # training days shown before the test window
plots_dir = os.path.join(os.path.dirname(os.path.abspath("AutoGluonTS.ipynb")), "plots_autogluon")
os.makedirs(plots_dir, exist_ok=True)

for sid in ts_raw.item_ids:
    fig, axes = plt.subplots(1, len(DS_ORDER), figsize=(18, 3), sharey=True)
    fig.suptitle(f"{sid} \u2014 Jan\u2013Mar 2025  |  AutoGluon-TS", fontsize=11, fontweight="bold")

    obs = test_raw.loc[sid]["DO_mgL"]

    for ax, ds_name in zip(axes, DS_ORDER):
        ds = DATASETS[ds_name]
        if sid not in ds["test"].item_ids:
            ax.set_visible(False)
            continue

        # Training context: last CONTEXT_DAYS before the test cutoff
        y_ctx = ds["train"].loc[sid]["DO_mgL"].iloc[-(24 * CONTEXT_DAYS):]
        ax.plot(y_ctx.index, y_ctx.values, color="gray", lw=1,
                alpha=0.7, label="Train (context)")

        ax.plot(obs.index, obs.values, color="steelblue", lw=1.5, label="Observed (raw)")

        for model_name, fc in forecasts[ds_name].items():
            if sid in fc.item_ids:
                pred_vals = fc.loc[sid]["mean"]
                ax.plot(pred_vals.index, pred_vals.values, lw=1,
                        linestyle="--", label=model_name)

        if sid in prophet_forecasts.get(ds_name, {}):
            fc_p = prophet_forecasts[ds_name][sid].reindex(obs.index)
            ax.plot(fc_p.index, fc_p.values, lw=1, linestyle=":", label="Prophet")

        sarima_fc = sarima_forecasts.get(ds_name, {}).get(sid)
        if sarima_fc is not None:
            fc_s = sarima_fc.reindex(obs.index)
            ax.plot(fc_s.index, fc_s.values, lw=1, linestyle="-.", label="SARIMA")

        ax.axvline(CUTOFF, color="black", lw=0.8, linestyle=":")
        ax.set_title(DS_LABELS[ds_name], fontsize=9)
        ax.set_ylabel("DO (mg/L)" if ax is axes[0] else "")
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
        ax.tick_params(axis="x", labelrotation=30, labelsize=7)

    plt.tight_layout()
    fig.savefig(
        os.path.join(plots_dir, f"{sid}_forecast.png"),
        dpi=300, bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)

print(f"Plots saved to: {plots_dir}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# FINAL CONSOLIDATED TABLE — all models × Baseline I / II / Hierarchical Filling
# ══════════════════════════════════════════════════════════════════════════════

all_metrics = pd.concat(
    [metrics_df, prophet_df, sarima_df],
    ignore_index=True,
)

ds_labels = {
    "baseline_I" : "Baseline I",
    "baseline_II": "Baseline II",
    "complete_series": "Hierarchical Filling",
}
all_metrics["Dataset"] = all_metrics["Dataset"].map(ds_labels).fillna(all_metrics["Dataset"])

metrics_order = ["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"]
dataset_order = ["Baseline I", "Baseline II", "Hierarchical Filling"]

summary = (
    all_metrics
    .groupby(["Model", "Dataset"])[metrics_order]
    .mean()
    .round(4)
)

summary_pivot = summary.unstack("Dataset")
summary_pivot = summary_pivot.reindex(
    columns=pd.MultiIndex.from_product([metrics_order, dataset_order])
)
summary_pivot.columns = [f"{m} [{d}]" for m, d in summary_pivot.columns]

print("\n" + "═" * 90)
print("  FINAL TABLE — Mean by Model (all stations)")
print("═" * 90)
display(summary_pivot)

csv_final = os.path.join(data_dir, "AutoGluonTS_metrics_FINAL.csv")
summary_pivot.to_csv(csv_final, sep=";", decimal=",")
print(f"\n  → Saved: {csv_final}")

# ── Full table by station and model ──────────────────────────────────────────
print("\n" + "═" * 90)
print("  FINAL TABLE — By Station and Model")
print("═" * 90)

full_pivot = all_metrics.pivot_table(
    index=["Model", "Station"],
    columns="Dataset",
    values=metrics_order,
    aggfunc="first",
).round(4)

full_pivot = full_pivot.reindex(
    columns=pd.MultiIndex.from_product([metrics_order, dataset_order])
)
full_pivot.columns = [f"{m} [{d}]" for m, d in full_pivot.columns]

display(full_pivot)

csv_full = os.path.join(data_dir, "AutoGluonTS_metrics_FINAL_completa.csv")
full_pivot.to_csv(csv_full, sep=";", decimal=",")
print(f"  → Saved: {csv_full}")

In [ ]:
# ── Per-station table: Baseline I / II vs Hierarchical Filling ─────────────────────

station_ag = pd.read_csv(
    os.path.join(data_dir, "AutoGluonTS_metrics_FINAL_completa.csv"),
    sep=";", decimal=",",
)

bI_cols   = [c for c in station_ag.columns if "[Baseline I]"       in c]
bII_cols  = [c for c in station_ag.columns if "[Baseline II]"      in c]
trat_cols = [c for c in station_ag.columns if "[Hierarchical Filling]"   in c]
metric_cols = bI_cols + bII_cols + trat_cols

styled_ag = (
    station_ag.style
    .format({c: "{:.4f}" for c in metric_cols})
    .set_caption("AutoGluonTS — Metrics by Station: Baseline I / Baseline II / Hierarchical Filling")
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold")]},
        {"selector": "th",      "props": [("text-align","center")]},
        {"selector": "td",      "props": [("text-align","right")]},
    ])
)

_hb = {"R²", "DA", "MDA"}
styled_ag = (
    styled_ag
    .highlight_min(subset=[c for c in trat_cols if not any(m in c for m in _hb)], color="lightgreen", axis=0)
    .highlight_max(subset=[c for c in trat_cols if     any(m in c for m in _hb)], color="lightgreen", axis=0)
)

display(styled_ag)
print(f"→ Source: AutoGluonTS_metrics_FINAL_completa.csv  ({station_ag.shape[0]} rows)")

In [ ]:
# ── Statistical tests: Baselines vs Hierarchical Filling — by Model × Metric ────────
# Two paired comparisons: (Baseline I vs Treated) and (Baseline II vs Treated).
# H0: no difference. Mean Δ > 0 → Treated improved over the baseline.

from scipy import stats
import numpy as np

station_ag = pd.read_csv(
    os.path.join(data_dir, "AutoGluonTS_metrics_FINAL_completa.csv"),
    sep=";", decimal=",",
)

_METRIC_NAMES = ["MASE", "RMSSE", "R²", "DA", "MDA", "MAED", "CETrend"]
_COMPARISONS  = [
    ("Baseline I",  "[Baseline I]",      "[Hierarchical Filling]"),
    ("Baseline II", "[Baseline II]",     "[Hierarchical Filling]"),
]

def _run_paired_tests(station_df, comparisons, metric_names):
    rows = []
    for model, grp in station_df.groupby("Model"):
        for comp_label, ref_tag, trat_tag in comparisons:
            for mname in metric_names:
                ref_col  = f"{mname} {ref_tag}"
                trat_col = f"{mname} {trat_tag}"
                if ref_col not in grp.columns or trat_col not in grp.columns:
                    continue
                ref_vals  = grp[ref_col].values
                trat_vals = grp[trat_col].values
                diff      = ref_vals - trat_vals   # > 0 → Treated improved

                t_stat, t_p = stats.ttest_rel(ref_vals, trat_vals)
                try:
                    w_stat, w_p = stats.wilcoxon(diff, zero_method="wilcox", alternative="two-sided")
                except ValueError:
                    w_stat, w_p = np.nan, np.nan

                rows.append({
                    "Comparison" : comp_label,
                    "Model"      : model,
                    "Metric"     : mname,
                    "n"          : len(diff),
                    "Mean Δ"     : diff.mean(),
                    "Mean Δ (%)" : (diff / np.abs(ref_vals) * 100).mean(),
                    "t-stat"     : t_stat,
                    "t p-value"  : t_p,
                    "W-stat"     : w_stat,
                    "W p-value"  : w_p,
                    "t sig."     : "✓" if t_p < 0.05 else "",
                    "W sig."     : "✓" if (not np.isnan(w_p) and w_p < 0.05) else "",
                })
    return pd.DataFrame(rows)

tests_df = _run_paired_tests(station_ag, _COMPARISONS, _METRIC_NAMES)

def _color_pval(v):
    if isinstance(v, float) and not np.isnan(v):
        if v < 0.05: return "background-color: #c6efce; color: #276221"
        if v < 0.10: return "background-color: #ffeb9c; color: #9c6500"
    return ""

def _color_delta(v):
    if isinstance(v, float):
        return "color: green" if v > 0 else ("color: red" if v < 0 else "")
    return ""

dcols = ["Comparison","Model","Metric","n","Mean Δ","Mean Δ (%)","t-stat","t p-value","W-stat","W p-value","t sig.","W sig."]

styled = (
    tests_df[dcols].style
    .format({
        "Mean Δ":     "{:+.4f}",
        "Mean Δ (%)": "{:+.2f}%",
        "t-stat":      "{:.3f}",
        "t p-value":   "{:.4f}",
        "W-stat":      lambda v: f"{v:.1f}" if not (isinstance(v, float) and np.isnan(v)) else "—",
        "W p-value":   lambda v: f"{v:.4f}" if not (isinstance(v, float) and np.isnan(v)) else "—",
    })
    .applymap(_color_pval,  subset=["t p-value", "W p-value"])
    .applymap(_color_delta, subset=["Mean Δ", "Mean Δ (%)"])
    .set_caption(
        "Paired tests: Baselines vs Hierarchical Filling by Model × Metric  "
        "(Δ > 0 = Treated improved | green = p < 0.05 | yellow = p < 0.10)"
    )
    .set_table_styles([
        {"selector": "caption", "props": [("font-size","13px"),("font-weight","bold")]},
        {"selector": "th",      "props": [("text-align","center")]},
        {"selector": "td",      "props": [("text-align","right")]},
    ])
)

display(styled)

csv_tests = os.path.join(data_dir, "AutoGluonTS_testes_estatisticos.csv")
tests_df.to_csv(csv_tests, index=False, sep=";", decimal=",")
print(f"→ Saved: {csv_tests}")
print()
print("Note: n=6 stations → limited power (min Wilcoxon p ≈ 0.125). Use Mean Δ (%) as practical effect size.")

In [ ]:
# ── Timing summary — Fit and Prediction by Model × Dataset ───────────────────
all_t = pd.concat([metrics_df, prophet_df, sarima_df], ignore_index=True)
all_t["Dataset"] = all_t["Dataset"].map({
    "baseline_I" : "Baseline I",
    "baseline_II": "Baseline II",
    "complete_series": "Hierarchical Filling",
}).fillna(all_t["Dataset"])

time_summary = (
    all_t.groupby(["Model", "Dataset"])[["Fit_s", "Pred_s"]]
    .mean()
    .round(2)
    .unstack("Dataset")
)
time_summary = time_summary.reindex(
    columns=pd.MultiIndex.from_product(
        [["Fit_s", "Pred_s"], ["Baseline I", "Baseline II", "Hierarchical Filling"]]
    )
)
time_summary.columns = [f"{m} [{d}]" for m, d in time_summary.columns]
time_summary.loc["MEAN"] = time_summary.mean()

print("\n" + "═"*80)
print("  Mean time per station — Fit (s) and Prediction (s)")
print("═"*80)
display(time_summary.round(2))

csv_time = os.path.join(data_dir, "AutoGluonTS_tempos.csv")
time_summary.round(2).to_csv(csv_time, sep=";", decimal=",")
print(f"  → Saved: {csv_time}")